<a href="https://colab.research.google.com/github/Thilac01/Point_Cloud/blob/main/Point_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import os

In [ ]:
file_path = "room_scan1.pcd"

if not os.path.exists(file_path):
    print(f"ERROR: '{file_path}' not found. Please upload it to Colab!")
else:
    # Load the point cloud
    pcd = o3d.io.read_point_cloud(file_path)

    if pcd.is_empty():
        print("ERROR: The point cloud is empty or corrupted.")
    else:
        print(f"Success! Loaded {len(pcd.points)} points from {file_path}.")

In [ ]:
# Create a coordinate frame (X=Red, Y=Green, Z=Blue) to help orient the view
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])

print("Rendering original 3D point cloud...")

# Visualize inline
o3d.visualization.draw_plotly(
    [pcd, coordinate_frame],
    window_name="Original Point Cloud",
    width=1000,
    height=600
)

#View All the X,Y,Z corrdinate only

In [ ]:


pcd = o3d.io.read_point_cloud("room_scan1.pcd")

points = np.asarray(pcd.points)
print(points)


print(f"Orignal data size: {len(points)}")

In [ ]:
# Set voxel size (e.g., 0.05 = 5 centimeters if your data is in meters)
voxel_size = 0.05

print(f"Original point count: {len(pcd.points)}")

# Perform downsampling
downpcd = pcd.voxel_down_sample(voxel_size=voxel_size)

print(f"Downsampled point count: {len(downpcd.points)}")

# Calculate the reduction
reduction = 100 - ((len(downpcd.points) / len(pcd.points)) * 100)
print(f"Data reduced by: {reduction:.2f}%")

In [ ]:
print("Rendering downsampled 3D point cloud...")

o3d.visualization.draw_plotly(
    [downpcd, coordinate_frame],
    window_name="Downsampled Point Cloud",
    width=1000,
    height=600
)

In [ ]:
# 1. Convert the downsampled Open3D object into a raw NumPy array
points_3d = np.asarray(downpcd.points)
print(f"3D Array Shape: {points_3d.shape}")

# 2. Initialize PCA to force the data into exactly 2 dimensions
pca = PCA(n_components=2)

# 3. Transform the data
points_2d = pca.fit_transform(points_3d)

print(f"2D Array Shape: {points_2d.shape}")
print(f"Geometry preserved (Variance retained): {sum(pca.explained_variance_ratio_) * 100:.2f}%")

In [ ]:
plt.figure(figsize=(10, 10))

# Scatter plot: X is PCA Component 1, Y is PCA Component 2
# s=1 sets the dot size to be very small for a cleaner map
plt.scatter(points_2d[:, 0], points_2d[:, 1], s=1, c='black', alpha=0.6)

plt.title("2D Floor Plan Projection (Flattened via PCA)")
plt.xlabel("Component 1 (Length/Width)")
plt.ylabel("Component 2 (Length/Width)")

# KEEP THIS: It ensures your room doesn't look stretched out like a funhouse mirror
plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

In [ ]:
import alphashape
from shapely.geometry import Polygon, MultiPolygon
from rdp import rdp
import numpy as np
import matplotlib.pyplot as plt

# 1. Create Alpha Shape
alpha = 0.5
alpha_shape = alphashape.alphashape(points_2d, alpha)

# 2. Fix: Check if it's a MultiPolygon or Polygon
if isinstance(alpha_shape, MultiPolygon):
    # If there are multiple parts, grab the one with the largest area
    # This ignores small noisy 'islands' that might have been detected
    alpha_shape = max(alpha_shape.geoms, key=lambda a: a.area)

# 3. Get exterior coordinates
hull_pts = np.array(alpha_shape.exterior.coords)

# 4. Simplify: Douglas-Peucker algorithm
corners = rdp(hull_pts, epsilon=0.8)

# 5. Plot
plt.figure(figsize=(10, 10))
plt.scatter(points_2d[:, 0], points_2d[:, 1], s=1, c='gray', alpha=0.3)
plt.plot(np.array(corners)[:, 0], np.array(corners)[:, 1], 'r-', linewidth=2, label="Wall Perimeter")
plt.scatter(np.array(corners)[:, 0], np.array(corners)[:, 1], s=100, c='red', label="Corner Points")

plt.title("Robust Irregular Room Corner Detection")
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()